# Ценовой блок ДОМ.РФ (актуальные ряды); Росреестр исключён 21.09

Источник: `sandbox_datasets/dom-price-deals-20260921.xlsx` (VPS → ВМ, 2026-09-21).
15 метрик, 30 224 наблюдения. Листы: **Метаданные** (с колонкой «ЕДИНИЦА — ваш ввод»), **Данные (long)**, **Max регионы + RF**, **Сводка RF квартал**.

Контекст: каталог показателей (этап 3), единицы измерения заполняет владелец в XLS; после заполнения значения возвращаются в `derived.catalog_facets.unit_canonical`.

In [ ]:
import pandas as pd

XLS = '/home/ubuntu/sandbox_datasets/dom-price-deals-20260921.xlsx'
meta = pd.read_excel(XLS, sheet_name='Метаданные')
long = pd.read_excel(XLS, sheet_name='Данные (long)')
rfq = pd.read_excel(XLS, sheet_name='Сводка ДОМ.РФ (по регионам)')
print('метрик:', len(meta), '| наблюдений:', len(long), '| сводка:', len(rfq))
meta.head(16)

## Метаданные — заполните колонку «ЕДИНИЦА (ваш ввод)»
unit_id=18 означает «не определена в БД» — ваша колонка закроет пробел каталога.

## Файл-источник ДОМ.РФ (raw)
Скопирован из репозитория: `raw/domrf/price_index/` → ВМ `sandbox_datasets/raw_domrf_price_index/`. CSV `domrf_priceindex_all_regions.csv` — первоисточник трёх активных рядов: index_value→itzdr, mom_pct→itzdrpm, n_deals→csdr (5 025 строк: регион × месяц с 2021-01). Рядом: domrf_deals.xlsx и domrf_changes.xlsx (срезы сделок и изменений).

In [ ]:
# Файл-источник (загружен вручную 12.09, VPS raw/domrf/price_index → ВМ)
raw = pd.read_csv('/home/ubuntu/sandbox_datasets/raw_domrf_price_index/domrf_priceindex_all_regions.csv')
print('raw-CSV:', len(raw), 'строк; регионы:', raw.region.nunique(), '| колонки:', list(raw.columns))
raw.tail(3)

In [2]:
meta[['metric_code','Название','Частота','Свежесть','Глубина (точек)','ЕДИНИЦА (ваш ввод)']]

,metric_code,Название,Частота,Свежесть,Глубина (точек),ЕДИНИЦА (ваш ввод)
0,itzdr,Индекс цен на жильё ДОМ.РФ,кварт,fresh,5025,NaN
1,itzdrpm,"Индекс цен на жильё ДОМ.РФ, % к предыдущему ме...",кварт,fresh,5025,NaN
2,csdr,"Число сделок, ДОМ.РФ",кварт,fresh,5025,NaN
3,ksdrk,"Количество сделок ДКП, регион, квартал",кварт,fresh,854,NaN
4,mtdrk,"Медианная цена ДКП за м², регион, квартал",кварт,fresh,854,NaN
5,mtsrk,"Медианная цена сделки за м², регион, квартал",кварт,fresh,854,NaN
6,mtdrk_2,"Медианная цена ДДУ за м², регион, квартал",кварт,fresh,840,NaN
7,msark,"Медианная ставка аренды за м², регион, квартал",кварт,fresh,833,NaN
8,mpazrk,"Медианная площадь арендуемого жилья, регион, к...",кварт,fresh,833,NaN
9,kdark,"Количество договоров аренды, регион, квартал",кварт,fresh,833,NaN


## ⚠️ Росреестр исключён из основной БД (21.09)
Медианные цены (mtsrk, mtdrk, mtdrk_2), сделки (ksdrk, ksdrk_2, ksddrk), аренда (kdark, msark, kdark_2, mpazrk) — source=rosreestr, исключены владельцем: скачки по регионам, неинформативны. Полный дамп: `data/rosreestr_separate.db` (sqlite, репо VPS). Остаются **активные ряды**: itzdr, itzdrpm, csdr, tcnkodr (ДОМ.РФ), vsdr, stkpk (ЦБ).

Лист «Сводка ДОМ.РФ (по регионам)» — median/max по ~74 регионам на период.

## Max по регионам + RF — лист удалён из XLS
Все ряды этого листа — Росреестр (исключён). Данные: `data/rosreestr_separate.db`.

## Динамика медианных цен по регионам
Разворот long → pivot по периодам для ценовых рядов.

In [ ]:
# Активный ряд: itzdr (индекс цен на жильё ДОМ.РФ) по регионам
import pandas as pd, psycopg
con = psycopg.connect('host=localhost port=5432 dbname=research_wiki user=wiki')
df = pd.read_sql('''SELECT o.period_start::date p, r.name_ru region, o.value
FROM core.observation_v2 o JOIN core.region r USING(region_id)
WHERE o.metric_id=1142 AND o.region_id>1 ORDER BY o.period_start, o.value DESC''', con)
pivot = df.pivot_table(index='p', columns='region', values='value', aggfunc='first')
pivot.iloc[:, :12]

## После заполнения единиц
Экспортируйте лист «Метаданные» обратно в `~/sandbox_datasets/` — агент заберёт файл, проверит колонку и зальёт единицы в `derived.catalog_facets.unit_canonical` + вернёт XLS в репозиторий.

## Сводка ДОМ.РФ (по регионам): median/max на период

In [ ]:
rfq.sort_values('period')